# Business Program Management Analysis

"
            "This notebook explores a synthetic dataset of projects across several departments.
"
            "The dataset was generated to reflect realistic patterns in project management such as budget overruns,
"
            "schedule delays, risk assessments, and outcomes like success and ROI.
"
            "We'll perform an end-to-end analysis involving:
"
            "
"
            "* Loading and inspecting the dataset
"
            "* Cleaning and feature engineering
"
            "* Exploratory data analysis with visualisations
"
            "* Classification modelling to predict project success
"
            "* Regression modelling to estimate return on investment (ROI)
"
            "* Discussion of results and next steps
"
            "
"
            "> **Note:** The data and models in this notebook are synthetic and are intended for educational purposes."
        )
    )

    # Import libraries and load data
    nb.cells.append(
        new_code_cell(
            import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (classification_report, confusion_matrix,
                         accuracy_score, roc_auc_score, mean_squared_error, r2_score)

# Set plotting style
sns.set(style='whitegrid', context='notebook')

# Load dataset
df = pd.read_csv('business_program_management_dataset.csv')

## Dataset Overview

"
            "Let's start by looking at the first few rows of the dataset to understand its structure."
        )
    )
    nb.cells.append(
        new_code_cell("df.head()")
    )

    nb.cells.append(
        new_markdown_cell(
            "We'll also check basic information about the columns, including data types and missing values."
        )
    )
    nb.cells.append(
        new_code_cell("df.info()")
    )

    nb.cells.append(
        new_code_cell("df.describe(include='all')")
    )

    # Feature engineering explanation
    nb.cells.append(
        new_markdown_cell(
            For our modelling tasks we need to prepare the data appropriately:

* **Categorical variables** (e.g. priority, department, program_type) will be one-hot encoded.
* **Numerical variables** will be scaled or left as-is depending on the model.
* We'll use the following target variables: 
  * `success` (binary classification).
  * `ROI` (regression).

## Exploratory Data Analysis

### Distribution of Numerical Variables

In [ ]:
numerical_cols = ['planned_duration', 'planned_budget', 'actual_budget',
                             'budget_overrun_pct', 'team_size', 'risk_level',
                             'complexity', 'manager_experience', 'scope_changes',
                             'delayed_tasks', 'resource_utilization_pct',
                             'schedule_overrun_pct', 'ROI', 'customer_satisfaction']
for col in numerical_cols:
    plt.figure(figsize=(6, 3))
    sns.histplot(df[col], kde=True, bins=30)
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()


### Relationship Between Key Factors and Success

In [ ]:
# Visualise categorical features vs success
categorical_cols = ['priority', 'department', 'program_type']
for col in categorical_cols:
    plt.figure(figsize=(6, 4))
    sns.countplot(data=df, x=col, hue='success')
    plt.title(f'{col} vs Success')
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.legend(title='Success', loc='upper right')
    plt.tight_layout()
    plt.show()

+
# Numerical relationships
plt.figure(figsize=(6, 4))
sns.boxplot(data=df, x='success', y='planned_budget')
plt.title('Planned Budget vs Success')
plt.show()

plt.figure(figsize=(6, 4))
sns.boxplot(data=df, x='success', y='risk_level')
plt.title('Risk Level vs Success')
plt.show()

plt.figure(figsize=(6, 4))
sns.boxplot(data=df, x='success', y='manager_experience')
plt.title('Manager Experience vs Success')
plt.show()


### Correlation Heatmap

In [ ]:
# Compute correlation matrix for numeric features

corr = df[numerical_cols + ['success']].corr()
+
plt.figure(figsize=(10, 8))

sns.heatmap(corr, cmap='coolwarm', annot=False, fmt='.2f', center=0)
+
plt.title('Correlation Heatmap')
+
plt.show()
+

## Predicting Project Success (Classification)

In [ ]:
# Separate features and target for classification
X_cls = df.drop(columns=['success', 'ROI'])
y_cls = df['success']

# Identify categorical and numerical columns
categorical_features = ['priority', 'department', 'program_type']
numeric_features = [
    'planned_duration', 'planned_budget', 'actual_budget', 'budget_overrun_pct',
    'team_size', 'risk_level', 'complexity', 'manager_experience',
    'scope_changes', 'delayed_tasks', 'resource_utilization_pct', 'schedule_overrun_pct',
    'customer_satisfaction'
]

# Preprocess: one-hot encode categoricals, leave numerics as-is
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first'), categorical_features),
        ('num', 'passthrough', numeric_features),
    ]
)

# Create train-test split
X_train, X_test, y_train, y_test = train_test_split(X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls)

# Pipeline with logistic regression
log_reg_clf = Pipeline(
    steps=[('preprocessor', preprocessor),
           ('classifier', LogisticRegression(max_iter=500))]
)

log_reg_clf.fit(X_train, y_train)
y_pred_lr = log_reg_clf.predict(X_test)
y_prob_lr = log_reg_clf.predict_proba(X_test)[:, 1]

print('Logistic Regression Accuracy:', accuracy_score(y_test, y_pred_lr))
print('Logistic Regression ROC AUC:', roc_auc_score(y_test, y_prob_lr))
print('
Classification Report (Logistic Regression):')
print(classification_report(y_test, y_pred_lr))

# Pipeline with Random Forest classifier
rf_clf = Pipeline(
    steps=[('preprocessor', preprocessor),
           ('classifier', RandomForestClassifier(n_estimators=200, random_state=42))]
)

rf_clf.fit(X_train, y_train)
y_pred_rf = rf_clf.predict(X_test)
y_prob_rf = rf_clf.predict_proba(X_test)[:, 1]

print('Random Forest Accuracy:', accuracy_score(y_test, y_pred_rf))
print('Random Forest ROC AUC:', roc_auc_score(y_test, y_prob_rf))
print('
Classification Report (Random Forest):')
print(classification_report(y_test, y_pred_rf))

# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(confusion_matrix(y_test, y_pred_lr), annot=True, fmt='d', ax=axes[0], cmap='Blues')
axes[0].set_title('Confusion Matrix - Logistic Regression')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt='d', ax=axes[1], cmap='Greens')
axes[1].set_title('Confusion Matrix - Random Forest')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()


## Predicting Return on Investment (Regression)

In [ ]:
# Separate features and target for regression
X_reg = df.drop(columns=['ROI'])
y_reg = df['ROI']

categorical_features_reg = ['priority', 'department', 'program_type']
numeric_features_reg = [
    'planned_duration', 'planned_budget', 'actual_budget', 'budget_overrun_pct',
    'team_size', 'risk_level', 'complexity', 'manager_experience',
    'scope_changes', 'delayed_tasks', 'resource_utilization_pct',
    'schedule_overrun_pct', 'customer_satisfaction', 'success'
]

preprocessor_reg = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first'), categorical_features_reg),
        ('num', 'passthrough', numeric_features_reg),
    ]
)

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# Linear Regression model
lin_reg = Pipeline(
    steps=[('preprocessor', preprocessor_reg),
           ('regressor', LinearRegression())]
)

lin_reg.fit(X_train_reg, y_train_reg)
y_pred_lr_reg = lin_reg.predict(X_test_reg)

rmse_lin = np.sqrt(mean_squared_error(y_test_reg, y_pred_lr_reg))
r2_lin = r2_score(y_test_reg, y_pred_lr_reg)

print('Linear Regression RMSE:', rmse_lin)
print('Linear Regression R^2:', r2_lin)

# Random Forest Regressor
rf_reg = Pipeline(
    steps=[('preprocessor', preprocessor_reg),
           ('regressor', RandomForestRegressor(n_estimators=200, random_state=42))]
)

rf_reg.fit(X_train_reg, y_train_reg)
y_pred_rf_reg = rf_reg.predict(X_test_reg)

rmse_rf = np.sqrt(mean_squared_error(y_test_reg, y_pred_rf_reg))
r2_rf = r2_score(y_test_reg, y_pred_rf_reg)

print('Random Forest Regressor RMSE:', rmse_rf)
print('Random Forest Regressor R^2:', r2_rf)



## Conclusions and Next Steps

In this notebook we generated insights into the factors contributing to project success
and return on investment. Through classification modelling we identified which features
are predictive of success, and through regression modelling we quantified the drivers of ROI.
Random Forest models generally performed better due to their ability to capture non-linear
relationships and interactions.

Possible extensions to this analysis include:

* Performing feature importance analysis to rank the most influential variables.
* Tuning hyperparameters using cross-validation.
* Investigating time-dependent patterns by incorporating project timelines.
* Adding clustering or segmentation to identify distinct project archetypes.
